<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/08%20-%20Avaliacao%20Modulo%201%20Motor%20de%20Intertravamento%20e%20Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Regras de Diagnóstico

Neste notebook implementamos a arquitetura de **Base de Conhecimento Industrial** orientada a objetos para a **Estação de Reabastecimento de Hidrogênio**. Estruturamos fatos, regras de produção em Cláusulas de Horn, mecanismos de verificação de consistência e exportação de relatórios estruturados.

As regras abaixo seguem os tags definidos na tabela de **Mapeamento de Variáveis de Processo** (Setor 100 - Armazenamento, Setor 200 - Condicionamento, Setor 300 - Dispensação).

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass, field
from typing import List, Set, Dict, Any, Optional
import time

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR" # 'SENSOR' ou 'INFERIDO'
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str      # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int      # 1 a 10 (10 = mais urgente)
    tempo_resposta_max_s: float
    procedimento_pop: str

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha"
    ):
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)

        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

    def obter_regras_por_fato(self, fato_nome: str) -> List[RegraDiagnostico]:
        return self._indice_antecedentes.get(fato_nome, [])

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        catalogo = []
        for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "POP": r.procedimento_pop
            })
        return catalogo

bc = BaseConhecimentoSCADA()

# ============================================================
# SETOR 100: ARMAZENAMENTO (Tanques de Baixa, Média e Alta Pressão)
# ============================================================

# --- Tanque de Baixa Pressão ---
bc.adicionar_regra(
    "R-01", ["p1_1"], "SOBREPRESSAO_TANQUE_BAIXA",
    "Pressão do Tanque de Baixa Pressão acima do limite (PT-101 > 400 bar)", "CRÍTICA", 9, 1.5,
    "POP-ARM-01: Monitorar PSV-101 e preparar fechamento de XV-101"
)
bc.adicionar_regra(
    "R-02", ["SOBREPRESSAO_TANQUE_BAIXA", "v1_1"], "TRIP_TANQUE_BAIXA",
    "Corte de Segurança do Tanque de Baixa Pressão", "CRÍTICA", 10, 0.8,
    "POP-ARM-02: Fechar XV-101 e confirmar atuação de PSV-101"
)
bc.adicionar_regra(
    "R-03", ["g1_1"], "FUGA_H2_TANQUE_BAIXA",
    "Vazamento de Hidrogênio na área do Tanque de Baixa Pressão (AT-101)", "CRÍTICA", 10, 1.0,
    "POP-SST-01: Acionar ALM-101, isolar YV-104A e evacuar área"
)

# --- Tanque de Média Pressão ---
bc.adicionar_regra(
    "R-04", ["p1_2"], "SOBREPRESSAO_TANQUE_MEDIA",
    "Pressão do Tanque de Média Pressão acima do limite (PT-102 > 700 bar)", "CRÍTICA", 9, 1.5,
    "POP-ARM-03: Monitorar PSV-102 e preparar fechamento de XV-102"
)
bc.adicionar_regra(
    "R-05", ["SOBREPRESSAO_TANQUE_MEDIA", "v1_2"], "TRIP_TANQUE_MEDIA",
    "Corte de Segurança do Tanque de Média Pressão", "CRÍTICA", 10, 0.8,
    "POP-ARM-04: Fechar XV-102 e confirmar atuação de PSV-102"
)
bc.adicionar_regra(
    "R-06", ["g1_2"], "FUGA_H2_TANQUE_MEDIA",
    "Vazamento de Hidrogênio na área do Tanque de Média Pressão (AT-102)", "CRÍTICA", 10, 1.0,
    "POP-SST-02: Acionar ALM-101 e evacuar área"
)

# --- Tanque de Alta Pressão ---
bc.adicionar_regra(
    "R-07", ["p1_3"], "SOBREPRESSAO_TANQUE_ALTA",
    "Pressão do Tanque de Alta Pressão acima do limite (PT-103 > 1000 bar)", "CRÍTICA", 9, 1.5,
    "POP-ARM-05: Monitorar PSV-103 e preparar fechamento de XV-103"
)
bc.adicionar_regra(
    "R-08", ["SOBREPRESSAO_TANQUE_ALTA", "v1_3"], "TRIP_TANQUE_ALTA",
    "Corte de Segurança do Tanque de Alta Pressão", "CRÍTICA", 10, 0.8,
    "POP-ARM-06: Fechar XV-103 e confirmar atuação de PSV-103"
)
bc.adicionar_regra(
    "R-09", ["g1_3"], "FUGA_H2_TANQUE_ALTA",
    "Vazamento de Hidrogênio na área do Tanque de Alta Pressão (AT-103)", "CRÍTICA", 10, 1.0,
    "POP-SST-03: Acionar ALM-101, isolar YV-104B e evacuar área"
)

# --- Segurança Geral do Setor 100 ---
bc.adicionar_regra(
    "R-10", ["e1_1"], "PARADA_EMERGENCIA_GERAL",
    "Parada de Emergência acionada manualmente pelo operador (ESD-100)", "CRÍTICA", 10, 0.3,
    "POP-SIS-01: Fechar XV-101, XV-102 e XV-103; desenergizar YV-104A/B; acionar ALM-101"
)

# ============================================================
# SETOR 200: CONDICIONAMENTO (Pré-resfriamento)
# ============================================================

bc.adicionar_regra(
    "R-11", ["nc_201", "h3_1"], "BLOQUEIO_DISPENSACAO_TEMPERATURA",
    "Início de abastecimento solicitado com pré-resfriamento fora da faixa (TT-201 > -40°C)", "ALTA", 6, 2.0,
    "POP-COND-01: Impedir abertura de XV-301 até TT-201 atingir -40°C"
)

# ============================================================
# SETOR 300: DISPENSAÇÃO (Transferência ao Veículo)
# ============================================================

bc.adicionar_regra(
    "R-12", ["g3_1"], "FUGA_H2_DISPENSADOR",
    "Vazamento de Hidrogênio detectado na área do dispensador (AT-301)", "CRÍTICA", 10, 0.8,
    "POP-DISP-01: Fechar XV-301, acionar ALM-101 e isolar a área de abastecimento"
)
bc.adicionar_regra(
    "R-13", ["t3_1"], "SOBRETEMPERATURA_RECEPCAO_VEICULO",
    "Temperatura no ponto de recepção do veículo acima do limite seguro (TT-301 > 85°C)", "ALTA", 8, 1.0,
    "POP-DISP-02: Fechar XV-301 imediatamente e abortar o abastecimento"
)
bc.adicionar_regra(
    "R-14", ["p3_1"], "ABASTECIMENTO_CONCLUIDO",
    "Pressão de enchimento atinge o setpoint do veículo (PT-301 ≈ 700 bar)", "BAIXA", 2, 5.0,
    "POP-DISP-03: Fechar XV-301 normalmente e liberar o veículo"
)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (ESTAÇÃO DE H₂) ===")
print(formatar_tabela(bc.exportar_catalogo()))

assert len(bc.regras) == 14
assert len(bc.obter_regras_por_fato("p1_1")) >= 1
assert len(bc.obter_regras_por_fato("g3_1")) >= 1
print("\n[OK] Base de Conhecimento estruturada, indexada e validada com sucesso!")


=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (ESTAÇÃO DE H₂) ===
ID   | Prioridade | Severidade | SE (Antecedentes)                  | ENTÃO (Consequente)               | Diagnóstico                                                                            | POP                                                                                
-----+------------+------------+------------------------------------+-----------------------------------+----------------------------------------------------------------------------------------+------------------------------------------------------------------------------------
R-02 | 10         | CRÍTICA    | SOBREPRESSAO_TANQUE_BAIXA AND v1_1 | TRIP_TANQUE_BAIXA                 | Corte de Segurança do Tanque de Baixa Pressão                                          | POP-ARM-02: Fechar XV-101 e confirmar atuação de PSV-101                           
R-03 | 10         | CRÍTICA    | g1_1                               | FUGA_H2_TANQUE